In [1]:
from flask import Flask, render_template, request
import os
import cv2
import numpy as np
import tensorflow as tf
import pytesseract as pt
from tensorflow.keras.preprocessing.image import load_img, img_to_array

app = Flask(__name__)
BASE_PATH = os.getcwd()
UPLOAD_PATH = os.path.join(BASE_PATH, 'static/upload/')
MODEL_PATH = os.path.join(BASE_PATH, 'object_detection.h5')

if not os.path.exists(UPLOAD_PATH):
    os.makedirs(UPLOAD_PATH)

def predict_and_ocr(model, image_path):
    image = load_img(image_path)
    image_np = np.array(image, dtype=np.uint8)
    h, w, d = image_np.shape
    image1 = load_img(image_path, target_size=(224, 224))
    image_arr_224 = img_to_array(image1) / 255.0
    test_arr = image_arr_224.reshape(1, 224, 224, 3)
    coords = model.predict(test_arr)
    denorm = np.array([w, w, h, h])
    coords = coords * denorm
    coords = coords.astype(np.int32)
    xmin, xmax, ymin, ymax = coords[0]
    roi = image_np[ymin:ymax, xmin:xmax]
    text = pt.image_to_string(roi)
    return text.strip()

@app.route('/', methods=['GET', 'POST'])
def index():
    result = None
    if request.method == 'POST':
        upload_file = request.files['image_name']
        filename = upload_file.filename
        path_save = os.path.join(UPLOAD_PATH, filename)
        upload_file.save(path_save)
        model = tf.keras.models.load_model(MODEL_PATH)
        result = predict_and_ocr(model, path_save)
    return render_template('index.html', result=result)

if __name__ == "__main__":
    app.run(debug=True)

ModuleNotFoundError: No module named 'flask'